In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt
import hdbscan

### First need to scale the input data 

In [2]:
def scale_data(df):
    '''
    Function that scales data to be used for PCA decomposition later

    Parameters 
    ----------
        df : panda data frame
            A df of just numeric values
    '''   
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df)
    return(scaled)

In [3]:
def pca_cum_var(scaled, nc):
    '''
    Function that finds the cumulative variance of components (from PCA decomposition) from scaled data

    Parameters 
    ----------
        scaled : panda data frame
            A df of scaled values (output of scale_data())
        nc : int
            The number of components to keep from the PCA decomposition.
    '''
    
    pca = PCA(n_components = nc)
    Y = pca.fit(scaled)
    var_exp = Y.explained_variance_ratio_
    cum_var_exp = np.cumsum(var_exp)
    cum_var_exp_pct = [f"{x * 100:.1f}%" for x in cum_var_exp]
    return(cum_var_exp_pct)

# cv = pca_cum_var(data_scaled, 50)

# # Report the amount of variance explained by the first n_components 
# counter = 1
# for x in cv: 
#     print(f"Dimension {counter}:\t {x}")
#     counter += 1

In [ ]:
def pca_fit_transform(scaled, nc, df_index):
    '''
    Function that performs a PCA decomposition on scaled data

    Parameters 
    ----------
        scaled : panda data frame
            A df of scaled values (output of scale_data())
        nc : int
            The number of components to keep from the PCA decomposition. Must match the same number used in pca_cum_var()
        df_index : list
            A list of sample names to give the output data frame as index
    '''
    
    pca = PCA(n_components = nc)
    pca_table = pca.fit_transform(scaled)
    colnames = [f"PC{x}" for x in range(1, nc+1)]
    pca_df = pd.DataFrame(data = pca_table, columns = colnames)
    pca_df = pca_df.set_index(df_index)
    pca_dict = {}
    pca_dict["df"] = pca_df
    pca_dict["pcs"] = pca_table
    return(pca_dict)

In [5]:
def perform_tsne(pca_df, nc, df_index):
    '''
    Function that performs a tsne decomposition on pca data

    Parameters 
    ----------
        pca_df : panda data frame
            A df of the PCA-transformed values (output of pca_fit_transform)
        nc : int
            The number of components to use in the tsne calculation
        df_index : list
            A list of sample names to give the output data frame as index
    '''
    tsne = TSNE(n_components=nc, verbose=1, perplexity=30, n_iter=1000, random_state = 200)
    tsne_results = tsne.fit_transform(pca_df)
    tsne_df = pd.DataFrame(data = tsne_results[:, 0:2], columns = ['tSNE1', 'tSNE2'])
    tsne_df = tsne_df.set_index(df_index)
    return(tsne_df) 

In [6]:
def plot_tsne(df, col_choice, continuous_col = None):
    '''
    Function that visualizes the resulting t-SNE from perform_tsne()

    Parameters 
    ----------
        df : panda data frame
            A data frame including the metadata and t-SNE embeddings for each sample
        col_choice : str
            Choice of column to use for assigning categories in plot
    '''
    plt.figure.Figure(figsize=(16,10))

    if continuous_col is not None:
        cmap = sns.cubehelix_palette(rot=-.2, as_cmap=True)
        
        sns.scatterplot(
        x="tSNE1", y="tSNE2",
        hue = "ESTIMATE",
        palette=cmap,
        data=df,
        legend="full",
        alpha=0.5)    
    else:
        sns.scatterplot(
        x="tSNE1", y="tSNE2",
        hue=col_choice,
        palette=sns.color_palette("bright"),
        data=df,
        legend="full",
        alpha=0.5
    )

## Creating a t-SNE plot by first decomposing using PCA

##### Import files

In [ ]:

datafile = "CHANGE" # Replace "CHANGE" with the path to your csv file containing the data, with a header. Rows are genes and columns are samples
data = pd.read_csv(datafile, engine = "pyarrow")
data = data.set_index(data.columns[0])

metafile = "CHANGE" # Replace "CHANGE" with the path to your metadata csv file, with a header. Rows are sample names and columns are sample characteristics (subtype, sex, etc.). To be compatible with this example, we assume you only have one type of sample characteristic, so your input file is two columns: samples and the characteristic.
meta = pd.read_csv(metafile, engine = "pyarrow")
meta = meta.set_index(meta.columns[0])

Your datafile should look something like the following once imported.

<img src="./datafile_example.png" style="height: 150px;"/>

In [ ]:
print(data.head())

Your metafile should look something like the following once imported.

<img src="./metafile_example.png" style="height: 150px;"/>

In [ ]:
print(meta.head())

##### Scale the data

In [ ]:
data_scaled = scale_data(data.T)

Your data_scaled should now look something like the following.

<img src="./data_scaled_example.png" style="height: 150px;"/>

In [ ]:
print(data_scaled)

##### Perform PCA, then tSNE

In [ ]:
# PCA and t-SNE
PCA_out = pca_fit_transform(data_scaled, 50, data.T.index) 
PCA_df = PCA_out["df"] 
PCA_pcs = PCA_out["pcs"] 
tsne_res = perform_tsne(PCA_df, 2, data.index)
meta_new = pd.concat([meta, PCA_df, tsne_res], axis = 1) # merge all three tables based on their indeces
plot_tsne(meta_new, "CHANGE") # Replace "CHANGE" with the name of the column containing the sample characteristic (subtype, sex, etc.) you wish to color the samples by in the resulting tSNE.

## Using HDBSCAN to cluster your samples

In [ ]:
# Here, we will use the prinicpal components from the PCA analysis as our input to HDBSCAN
clusterer = hdbscan.HDBSCAN(min_cluster_size = 5) # The minimum number of samples to be considered a group is 5
clusterer.fit(PCA_df)

In [ ]:
PCA_df_w_HDBSCAN = PCA_df
PCA_df_w_HDBSCAN["HDBSCAN_label"] = clusterer.labels_
meta_w_HDBSCAN = pd.concat([meta, PCA_df_w_HDBSCAN["HDBSCAN_label"]], axis = 1) # Add the resulting label HDBSCAN found to the original metadata
print(meta_w_HDBSCAN)
# meta_w_HDBSCAN.to_csv("CHANGE") # Put the path you wish to output your results to here

## K means clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# First need to find the best number of clusters
silhouette_coefficients = []

kmeans_kwargs = {
    "init": "random",
    "n_init": 10,
    "max_iter": 300,
    "random_state": 42,
}

print(data_scaled)

# For each possible value of k (number of clusters), find the silhouette score, 
# which tells you how well-clustered your data is, with the value of 1 meaning
# the data clusters well and does not overlap with other clusters
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, **kmeans_kwargs)
    kmeans.fit(PCA_pcs)
    score = silhouette_score(PCA_pcs, kmeans.labels_)
    silhouette_coefficients.append(score)
    
import matplotlib.pyplot as plt
plt.style.use("fivethirtyeight")
plt.plot(range(2, 11), silhouette_coefficients)
plt.xticks(range(2, 11))
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Coefficient")
plt.show()

### Extract cluster assignments to samples

In [ ]:
# Assuming you have good clustering in the previous step (e.g. one of your values of k gave a result
# close to 1), put the value with the max k here to assign the cluster labels to samples
km = KMeans(n_clusters=CHANGE, **kmeans_kwargs).fit(data_scaled) # replace change with an integer representing the value of k you chose

cluster_map = pd.DataFrame()
cluster_map['data_index'] = data.index.values
cluster_map['cluster'] = km.labels_

print("Below, the first column is the cluster number and the second column is the number of samples assigned to that cluster")
print(cluster_map['cluster'].value_counts())

# cluster_map.to_csv("CHANGE") # Put the path you wish to output your results to here

## Hierarchical clustering (using agglomerative clustering specifically)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

agg_model = AgglomerativeClustering(n_clusters=2, metric='euclidean', linkage='ward')
clusters = agg_model.fit_predict(PCA_df)

cluster_map = pd.DataFrame()
cluster_map['data_index'] = PCA_df.index.values
cluster_map['cluster'] = agg_model.labels_
# cluster_map.to_csv("CHANGE") # Put the path you wish to output your results to here

Below, the first column is the cluster number and the second column is the number of samples assigned to that cluster"

In [ ]:
print(cluster_map)

Below, the first column is the cluster number and the second column is the number of samples assigned to that cluster

In [ ]:
print(cluster_map['cluster'].value_counts())